In [1]:
from leo_utils import arc_point_on_earth, compute_satellite_intersection_point_enu, compute_az_el_dist
import numpy as np
import json, gc, pandas as pd
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import importlib, minmax_solvers
importlib.reload(minmax_solvers)
from minmax_solvers import solve_game_bestresp_Q0_then_Q1
from numpy.linalg import cholesky, solve, eigh
from Joint_waterfilling import (
    dominant_precoder_from_cir,
    joint_waterfilling_from_cir,
)

# Physical constants and power settings.
c = 3e8  # speed of light (m/s)
fc = 10e9  # carrier frequency (Hz)
wavelength = c / fc
tx_power_dbm = 50
jam_power_dbm = 70
k = 1.38e-23  # Boltzmann constant
GT = 13
La = 5

# Static CIR export settings.
bandwidth = 200e6
cir_sampling_frequency = 1.0
cir_num_time_steps = 1

GT_linear_inv = 10 ** (-GT / 10)
La_linear = 10 ** (La / 10)
noise_power_watt = k * bandwidth * GT_linear_inv * La_linear
Tx_power_watt = 10 ** ((tx_power_dbm - 30) / 10)
Jam_power_watt = 10 ** ((jam_power_dbm - 30) / 10)
T_sys = 150.0
N0 = k * bandwidth * T_sys * La_linear
P0 = Tx_power_watt
P1 = Jam_power_watt


In [2]:
from sionna.rt import Receiver, Transmitter, PlanarArray, PathSolver, load_scene
import numpy as np
import vsat_dish_3gpp


def _as_2d_positions(rx_pos):
    rx_pos = np.asarray(rx_pos, dtype=float)
    if rx_pos.ndim == 1:
        if rx_pos.shape[0] != 3:
            raise ValueError("rx_pos must have shape (3,) or (K, 3)")
        rx_pos = rx_pos[None, :]
    if rx_pos.ndim != 2 or rx_pos.shape[1] != 3:
        raise ValueError("rx_pos must have shape (3,) or (K, 3)")
    return rx_pos


def _broadcast_velocity(velocity, count, name):
    if velocity is None:
        return np.zeros((count, 3), dtype=float)

    velocity = np.asarray(velocity, dtype=float)
    if velocity.ndim == 1:
        if velocity.shape[0] != 3:
            raise ValueError(f"{name} must have shape (3,) or ({count}, 3)")
        velocity = np.repeat(velocity[None, :], count, axis=0)

    if velocity.ndim != 2 or velocity.shape != (count, 3):
        raise ValueError(f"{name} must have shape (3,) or ({count}, 3)")
    return velocity


def compute_paths(tx_pos, rx_pos, tx_array, rx_array, tx_look_at, rx_look_at=None,
                  frequency=10e9, tx_velocity=None, rx_velocities=None,
                  max_depth=0, los=True, respect_rx_look_at=False):
    """
    Solve paths from one transmitter to one or more receivers.

    Time variation is controlled through tx/rx velocities. To obtain a
    time-varying CIR, call ``compute_cir(..., num_time_steps>1, sampling_frequency=...)``.

    Args:
        tx_pos:            (3,) transmitter position
        rx_pos:            (3,) or (K, 3) receiver positions
        tx_velocity:       (3,) transmitter velocity in ENU [m/s]
        rx_velocities:     (3,) or (K, 3) receiver velocities in ENU [m/s]
        respect_rx_look_at:
            False keeps the notebook's current RX pointing behavior.
            True makes each RX call ``look_at(rx_look_at)``.

    Returns:
        paths: Sionna RT ``Paths`` object
    """
    tx_pos = np.asarray(tx_pos, dtype=float)
    if tx_pos.shape != (3,):
        raise ValueError("tx_pos must have shape (3,)")

    rx_pos = _as_2d_positions(rx_pos)
    tx_velocity = _broadcast_velocity(tx_velocity, 1, "tx_velocity")[0]
    rx_velocities = _broadcast_velocity(rx_velocities, rx_pos.shape[0], "rx_velocities")

    scene = load_scene()
    scene.frequency = frequency
    scene.synthetic_array = True

    for tx_name in list(scene.transmitters):
        scene.remove(tx_name)
    for rx_name in list(scene.receivers):
        scene.remove(rx_name)

    scene.tx_array = tx_array
    tx = Transmitter(name="tx", position=tx_pos, display_radius=200)
    scene.add(tx)

    if isinstance(tx_look_at, str) and tx_look_at.lower() in ["up-z", "up", "z"]:
        tx_look_at = tx_pos + np.array([0.0, 0.0, 1.0])
    tx.look_at(tx_look_at)
    tx.velocity = tx_velocity.tolist()

    earth_radius_m = 6371e3
    legacy_rx_look_at = np.array([0.0, 0.0, -1.0 * earth_radius_m / 2.0])

    scene.rx_array = rx_array
    for i, (rx_pos_i, rx_vel_i) in enumerate(zip(rx_pos, rx_velocities)):
        rx = Receiver(name=f"rx{i}", position=rx_pos_i)
        scene.add(rx)
        if respect_rx_look_at and rx_look_at is not None:
            rx.look_at(rx_look_at)
        else:
            rx.look_at(legacy_rx_look_at)
        rx.velocity = rx_vel_i.tolist()

    solver = PathSolver()
    paths = solver(scene=scene,
                   max_depth=max_depth,
                   los=los,
                   synthetic_array=True)
    return paths


def compute_cir(tx_pos, rx_pos, tx_array, rx_array, tx_look_at, rx_look_at,
                frequency=10e9, tx_velocity=None, rx_velocities=None,
                sampling_frequency=1.0, num_time_steps=1,
                normalize_delays=False, max_depth=0, los=True,
                respect_rx_look_at=False, return_paths=False):
    """
    Compute CIR from one transmitter to one or more receivers.

    When ``num_time_steps > 1`` and non-zero velocities are provided, ``a_all``
    contains the time evolution across the last axis.

    Returns:
        a_all: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
        tau_all: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
    """
    paths = compute_paths(
        tx_pos=tx_pos,
        rx_pos=rx_pos,
        tx_array=tx_array,
        rx_array=rx_array,
        tx_look_at=tx_look_at,
        rx_look_at=rx_look_at,
        frequency=frequency,
        tx_velocity=tx_velocity,
        rx_velocities=rx_velocities,
        max_depth=max_depth,
        los=los,
        respect_rx_look_at=respect_rx_look_at,
    )

    a_all, tau_all = paths.cir(
        sampling_frequency=sampling_frequency,
        num_time_steps=num_time_steps,
        normalize_delays=normalize_delays,
        out_type="numpy",
    )

    if return_paths:
        return a_all, tau_all, paths
    return a_all, tau_all

jam_rows =6
jam_cols = 6
jam_antennas = jam_cols*jam_rows

sat_rows = 6
sat_cols = 6
sat_antennas = sat_cols*sat_rows

tx_rows = 6
tx_cols = 6
tx_antennas = tx_cols*tx_rows

tx_array = PlanarArray(num_rows=tx_rows, num_cols=tx_cols,
                        vertical_spacing=0.5, horizontal_spacing=0.5,
                        pattern="tr38901", polarization="V")
                        # pattern="iso", polarization="V")

jam_array = PlanarArray(num_rows=jam_rows, num_cols=jam_cols,  
                            vertical_spacing=0.5, horizontal_spacing=0.5,
                        #  pattern="vsat_dish",
                            pattern="tr38901",
                            polarization="V")

sat_array = PlanarArray(num_rows=sat_rows, num_cols=sat_cols,
                             vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

def collapse_channel(a_cir, t_idx=None):
    """
    Collapse channel tensor over the path axis.

    Input:
      a_cir: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]

    Output:
      If t_idx is None:
          H_t: (M, N, num_time_steps)
      else:
          H_t: (M, N)

      where M = num_rx*num_rx_ant, N = num_tx*num_tx_ant.
    """
    nr, nra, nt, nta, npaths, ntimes = a_cir.shape

    H_t = a_cir.sum(axis=-2)  # sum over paths -> [nr, nra, nt, nta, ntimes]

    if t_idx is None:
        H_t = H_t.reshape(nr * nra, nt * nta, ntimes)
        return H_t

    assert 0 <= t_idx < ntimes
    H_slice = H_t[..., t_idx]
    H_slice = H_slice.reshape(nr * nra, nt * nta)
    return H_slice


In [3]:

def make_tqdm_progress(total, desc, position=1, leave=False):
    bar = tqdm(total=total, desc=desc, position=position,
                  leave=leave, dynamic_ncols=True, miniters=1, mininterval=0.0)
    bar.refresh()  
    last_i = 0

    def _cb(i=None, total=None, metrics=None, ctx=None):
        nonlocal last_i
        if i is not None:
            di = int(i) - int(last_i)     
            if di > 0:
                bar.update(di)
                last_i = i
        if metrics:
            show = {k: (f"{v:.3e}" if ("res" in k or "err" in k) else f"{v:.4f}")
                    for k, v in metrics.items()}
            bar.set_postfix(show, refresh=True)  
        return False

    _cb.close = bar.close
    return _cb

In [4]:
# Load all 5-minute frames and rank satellites by shortest slant range per frame.
csv_path = "starlink_timeseries_5min_all.csv"
df = pd.read_csv(csv_path)

if "Rank" not in df.columns:
    df = df.sort_values(["Time", "Slant km"], ascending=[True, True]).copy()
    df["Rank"] = df.groupby("Time").cumcount() + 1
else:
    df = df.sort_values(["Time", "Rank"], ascending=[True, True]).copy()

df = df.reset_index(drop=True)
n_steps = df["Time"].nunique()
print(f"Source: {csv_path}")
print("Total steps:", n_steps)
print("Rows:", len(df))


Source: starlink_timeseries_5min_all.csv
Total steps: 30
Rows: 582


In [ ]:

# Generate TX positions on the ground

distances_km = [7]
azimuths_deg = np.linspace(0, 360, len(distances_km), endpoint=False)
gnd_positions = [np.array([0.0, 0.0, 0.0])]

for d_km, az in zip(distances_km, azimuths_deg):
    pos = arc_point_on_earth(d_km, az)
    gnd_positions.append(pos)
gnd_positions = np.array(gnd_positions)

for i, pos in enumerate(gnd_positions):
    print(f"TX{i}(m): {pos}")
    
# # Generate Sats positions 
# sat_orbit_m = 550e3
# angles = [(117.88, 40),(230.22, 89.9)]
# sat_positions = []
# delays_ms = []
# fspl_db = []
# frequency_hz = 10e9 
# wavelength = 3e8 / frequency_hz

# for az, el in angles:
#     pos, delay, dist = compute_satellite_intersection_point_enu(az, el, sat_orbit_m)
#     sat_positions.append(pos)
#     delays_ms.append(delay)
#     fspl = 20 * np.log10(4 * np.pi * dist / wavelength)
#     fspl_db.append(fspl)

# sat_positions = np.array(sat_positions)
# delays_ms = np.array(delays_ms)
# fspl_db = np.array(fspl_db)

# print("\nSatellite Pos [m]:\n", sat_positions)
# print("\nPropagation delays [ms]:\n", delays_ms)
# print("\nFree-space path loss [dB]:\n", fspl_db)

# delay_max = np.max(delays_ms) - np.min(delays_ms)
# print("\ndelay_max [ms]:\n", delay_max)
# Compute az/el/dist per TX-SAT

# for i, tx in enumerate(gnd_positions):
#     print(f"\nFrom TX{i}:")
#     for j, sat in enumerate(sat_positions):
#         az, el, dist, n_waves = compute_az_el_dist(sat, tx, frequency_hz)
#         print(f"  SAT{j}: az={az:.2f}°, el={el:.2f}°, dist={dist:.2f} m, λ count ≈ {n_waves:.2f}")
# csv_path = "starlink_timeseries_5min_all.csv"
# df_all = pd.read_csv(csv_path)

# vel_cols = ["vx_East (m/s)", "vy_North (m/s)", "vz_Up (m/s)"]
# speed_total_mps = np.sqrt((df_all[vel_cols].astype(float) ** 2).sum(axis=1))

# max_speed_idx = speed_total_mps.idxmax()
# max_speed_mps = speed_total_mps.loc[max_speed_idx]

# print("\\nMax total speed [m/s]:\\n", max_speed_mps)
# print(df_all.loc[max_speed_idx])

TX0(m): [0. 0. 0.]
TX1(m): [ 4.28626293e-13  6.99999859e+03 -3.84554976e+00]


In [6]:
# ---- JSON serializer: supports numpy / complex ----
class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (np.complexfloating, complex)):
            return {"__complex__": [float(o.real), float(o.imag)]}
        if isinstance(o, np.ndarray):
            return o.tolist()
        return super().default(o)


def _select_groups(groups, run_steps):
    if isinstance(run_steps, str):
        if run_steps.lower() != "all":
            raise ValueError("run_steps must be an integer or 'all'")
        return groups

    run_steps = int(run_steps)
    if run_steps <= 0:
        raise ValueError("run_steps must be positive")
    return groups[:run_steps]


def _select_topk(group_df, topk):
    group_df = group_df.sort_values("Rank").copy()
    if isinstance(topk, str):
        if topk.lower() != "all":
            raise ValueError("topk must be an integer or 'all'")
        return group_df

    topk = int(topk)
    if topk <= 0:
        raise ValueError("topk must be positive")
    return group_df.head(topk).copy()


run_steps = "all"  # int or "all"
sat_counts = [1, 3, 5]
out_path = "a_tx_a_jam_tau_5min_satcounts_1_3_5.jsonl"
overwrite_output = True

# Jammer precoding: solve one water-filling problem over the stacked top-k
# satellite receivers for each k in sat_counts.
jam_precoder_mode = "joint_waterfilling"
jam_precoder_num_streams = None
jam_precoder_t_idx = 0
save_jam_precoded_cir = True

FLUSH_EVERY = 1

groups = list(df.sort_values(["Time", "Rank"]).groupby("Time", sort=False))
groups_to_run = _select_groups(groups, run_steps)
total_steps = len(groups_to_run)
file_mode = "w" if overwrite_output else "a"

with open(out_path, file_mode, encoding="utf-8") as f_out:
    buffer = []

    for step_idx, (t, g) in enumerate(tqdm(groups_to_run, desc="Processing steps"), start=1):
        g_sorted = g.sort_values("Rank").copy()

        for k_sel in sat_counts:
            if len(g_sorted) < k_sel:
                continue

            gk = _select_topk(g_sorted, k_sel)
            if gk.empty:
                continue

            sat_positions = gk[["x_East (m)", "y_North (m)", "z_Up (m)"]].to_numpy()
            sat_velocities = gk[["vx_East (m/s)", "vy_North (m/s)", "vz_Up (m/s)"]].to_numpy()
            names = gk["Name"].tolist()
            ranks = gk["Rank"].astype(int).tolist()

            # a: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
            # tau: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
            a_tx, tau_tx = compute_cir(
                gnd_positions[0],
                sat_positions,
                tx_array,
                sat_array,
                np.asarray(gnd_positions[0]) + np.array([0.0, 0.0, 100.0]),
                gnd_positions[0],
                sampling_frequency=cir_sampling_frequency,
                num_time_steps=cir_num_time_steps,
                normalize_delays=False,
            )

            a_jam, tau_jam = compute_cir(
                gnd_positions[1],
                sat_positions,
                jam_array,
                sat_array,
                np.asarray(gnd_positions[1]) + np.array([0.0, 0.0, 100.0]),
                gnd_positions[0],
                sampling_frequency=cir_sampling_frequency,
                num_time_steps=cir_num_time_steps,
                normalize_delays=False,
            )

            jam_precoder_result = None
            if jam_precoder_mode == "joint_waterfilling":
                jam_precoder_result = joint_waterfilling_from_cir(
                    a_jam,
                    N0=N0,
                    P0=P1,
                    t_idx=jam_precoder_t_idx,
                    num_streams=jam_precoder_num_streams,
                    use_precoding=True,
                )
            elif jam_precoder_mode != "off":
                raise ValueError(
                    f"Unsupported jam_precoder_mode={jam_precoder_mode!r}. "
                    "Use 'off' or 'joint_waterfilling'."
                )

            a_jam_eff = (
                jam_precoder_result["a_eff"]
                if (jam_precoder_result is not None and save_jam_precoded_cir)
                else None
            )
            W_jam = jam_precoder_result["W_t"] if jam_precoder_result is not None else None
            w_jam = jam_precoder_result["w_t"] if jam_precoder_result is not None else None
            jam_active_modes = jam_precoder_result["active_modes"] if jam_precoder_result is not None else None
            jam_rate_bpcu = jam_precoder_result["rate_bpcu"] if jam_precoder_result is not None else None

            entry = {
                "time": (pd.Timestamp(t).isoformat() if not pd.isna(t) else None),
                "step_idx": step_idx,
                "source_csv": csv_path if "csv_path" in globals() else None,
                "k": int(len(names)),
                "topk": int(k_sel),
                "sat_names": names,
                "sat_ranks": ranks,
                "sat_slant_km": gk["Slant km"].to_numpy(),
                "sat_azimuth_deg": gk["Azimuth (°)"].to_numpy(),
                "sat_elevation_deg": gk["Elevation (°)"].to_numpy(),
                "sat_positions_enu_m": sat_positions,
                "sat_velocities_enu_mps": sat_velocities,
                "tx_position_enu_m": np.asarray(gnd_positions[0]),
                "jam_position_enu_m": np.asarray(gnd_positions[1]),
                "sampling_frequency_hz": cir_sampling_frequency,
                "num_time_steps": cir_num_time_steps,
                "a_tx": a_tx,
                "tau_tx": tau_tx,
                "a_jam": a_jam,
                "a_jam_eff": a_jam_eff,
                "tau_jam": tau_jam,
                "W_jam": W_jam,
                "w_jam": w_jam,
                "jam_precoder_mode": jam_precoder_mode,
                "jam_precoder_num_streams": jam_precoder_num_streams,
                "jam_active_modes": jam_active_modes,
                "jam_rate_bpcu": jam_rate_bpcu,
                "jam_use_precoding": bool(a_jam_eff is not None),
            }
            buffer.append(entry)

            del a_tx, tau_tx, a_jam, tau_jam, a_jam_eff, W_jam, w_jam, entry
            del jam_active_modes, jam_rate_bpcu, jam_precoder_result
            gc.collect()

        if (step_idx % FLUSH_EVERY == 0) or (step_idx == total_steps):
            for e in buffer:
                f_out.write(json.dumps(e, cls=NumpyEncoder) + "\n")
            f_out.flush()
            buffer.clear()
            gc.collect()
            print(f"[flush] wrote up to step {step_idx}/{total_steps}")

        del g_sorted
        gc.collect()

print(f"Static TX/JAM CIR saved to {out_path}")


Processing steps:   0%|          | 0/30 [00:00<?, ?it/s]

[flush] wrote up to step 1/30
[flush] wrote up to step 2/30
[flush] wrote up to step 3/30
[flush] wrote up to step 4/30
[flush] wrote up to step 5/30
[flush] wrote up to step 6/30
[flush] wrote up to step 7/30
[flush] wrote up to step 8/30
[flush] wrote up to step 9/30
[flush] wrote up to step 10/30
[flush] wrote up to step 11/30
[flush] wrote up to step 12/30
[flush] wrote up to step 13/30
[flush] wrote up to step 14/30
[flush] wrote up to step 15/30
[flush] wrote up to step 16/30
[flush] wrote up to step 17/30
[flush] wrote up to step 18/30
[flush] wrote up to step 19/30
[flush] wrote up to step 20/30
[flush] wrote up to step 21/30
[flush] wrote up to step 22/30
[flush] wrote up to step 23/30
[flush] wrote up to step 24/30
[flush] wrote up to step 25/30
[flush] wrote up to step 26/30
[flush] wrote up to step 27/30
[flush] wrote up to step 28/30
[flush] wrote up to step 29/30
[flush] wrote up to step 30/30
Static TX/JAM CIR saved to a_tx_a_jam_tau_5min_satcounts_1_3_5.jsonl


In [7]:
# --- Inspect one saved JSONL entry: metadata + array shapes ---
import json
from pathlib import Path
import numpy as np
import pandas as pd

inspect_path = "a_tx_a_jam_tau_5min_satcounts_1_3_5.jsonl"
inspect_step_idx = None   # e.g. 1; None keeps any step
inspect_time = None       # e.g. "2026-..."; exact match; None keeps any time
inspect_topk = 5          # choose 1, 3, 5, or None
inspect_match_index = -1  # -1 = last matching entry; 0 = first matching entry


def _complex_hook(obj):
    if "__complex__" in obj:
        real, imag = obj["__complex__"]
        return complex(real, imag)
    return obj


def resolve_jsonl_path(path):
    candidate = Path(path)
    if candidate.is_absolute():
        candidates = [candidate]
    else:
        candidates = [
            Path.cwd() / candidate,
            Path.cwd() / "Anti-jamming-convexset" / candidate,
            Path("/home/sj4025/my_project/Anti-jamming-convexset") / candidate,
            candidate,
        ]

    for item in candidates:
        if item.exists():
            return item

    raise FileNotFoundError(f"Could not find {path}. Tried: {[str(p) for p in candidates]}")


def _entry_matches(entry, step_idx=None, time=None, topk=None):
    if step_idx is not None and int(entry.get("step_idx")) != int(step_idx):
        return False
    if time is not None and str(entry.get("time")) != str(time):
        return False
    if topk is not None and int(entry.get("topk", entry.get("k"))) != int(topk):
        return False
    return True


def load_jsonl_entry(path, step_idx=None, time=None, topk=None, match_index=-1):
    resolved = resolve_jsonl_path(path)
    selected = None
    selected_line_idx = None
    match_count = 0

    with open(resolved, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            entry = json.loads(line, object_hook=_complex_hook)
            if not _entry_matches(entry, step_idx=step_idx, time=time, topk=topk):
                continue

            if match_index >= 0 and match_count == int(match_index):
                return entry, resolved, line_idx, match_count + 1

            selected = entry
            selected_line_idx = line_idx
            match_count += 1

    if selected is None:
        raise ValueError(
            "No matching JSONL entry found for "
            f"step_idx={step_idx}, time={time}, topk={topk}."
        )
    if match_index >= 0:
        raise ValueError(
            f"Requested match_index={match_index}, but only {match_count} matching entries exist."
        )

    return selected, resolved, selected_line_idx, match_count


def summarize_arrays(entry, keys):
    rows = []
    for key in keys:
        value = entry.get(key)
        if value is None:
            rows.append({"key": key, "present": key in entry, "shape": None, "dtype": None, "size": 0})
            continue

        arr = np.asarray(value)
        rows.append(
            {
                "key": key,
                "present": True,
                "shape": tuple(arr.shape),
                "ndim": arr.ndim,
                "dtype": str(arr.dtype),
                "size": int(arr.size),
                "nbytes_MB": float(arr.nbytes / 1e6),
            }
        )
    return pd.DataFrame(rows)


inspect_entry, inspect_resolved_path, inspect_line_idx, inspect_match_count = load_jsonl_entry(
    inspect_path,
    step_idx=inspect_step_idx,
    time=inspect_time,
    topk=inspect_topk,
    match_index=inspect_match_index,
)

metadata_keys = [
    "time",
    "step_idx",
    "topk",
    "k",
    "sat_names",
    "sat_ranks",
    "sampling_frequency_hz",
    "num_time_steps",
    "jam_precoder_mode",
    "jam_precoder_num_streams",
    "jam_active_modes",
    "jam_rate_bpcu",
    "jam_use_precoding",
]
array_keys = [
    "a_tx",
    "tau_tx",
    "a_jam",
    "tau_jam",
    "a_jam_eff",
    "W_jam",
    "w_jam",
    "a_eff",
    "W_t",
    "w_t",
    "sat_positions_enu_m",
    "sat_velocities_enu_mps",
]

print(
    f"Loaded line {inspect_line_idx + 1} from {inspect_resolved_path} "
    f"({inspect_match_count} matching entries scanned)."
)
display(pd.DataFrame([{"key": k, "value": inspect_entry.get(k)} for k in metadata_keys if k in inspect_entry]))
display(summarize_arrays(inspect_entry, array_keys))

# Convenience variables for direct notebook checks, e.g. a_jam_eff_sel.shape.
a_tx_sel = np.asarray(inspect_entry["a_tx"], dtype=complex) if inspect_entry.get("a_tx") is not None else None
tau_tx_sel = np.asarray(inspect_entry["tau_tx"], dtype=float) if inspect_entry.get("tau_tx") is not None else None
a_jam_sel = np.asarray(inspect_entry["a_jam"], dtype=complex) if inspect_entry.get("a_jam") is not None else None
tau_jam_sel = np.asarray(inspect_entry["tau_jam"], dtype=float) if inspect_entry.get("tau_jam") is not None else None
a_jam_eff_sel = np.asarray(inspect_entry["a_jam_eff"], dtype=complex) if inspect_entry.get("a_jam_eff") is not None else None
W_jam_sel = np.asarray(inspect_entry["W_jam"], dtype=complex) if inspect_entry.get("W_jam") is not None else None
w_jam_sel = np.asarray(inspect_entry["w_jam"], dtype=complex) if inspect_entry.get("w_jam") is not None else None
a_eff_sel = np.asarray(inspect_entry["a_eff"], dtype=complex) if inspect_entry.get("a_eff") is not None else None
W_t_sel = np.asarray(inspect_entry["W_t"], dtype=complex) if inspect_entry.get("W_t") is not None else None
w_t_sel = np.asarray(inspect_entry["w_t"], dtype=complex) if inspect_entry.get("w_t") is not None else None


Loaded line 90 from /home/sj4025/my_project/Anti-jamming-convexset/a_tx_a_jam_tau_5min_satcounts_1_3_5.jsonl (30 matching entries scanned).


,key,value
0,time,2026-01-15T19:12:46
1,step_idx,30
2,topk,5
3,k,5
4,sat_names,"[STARLINK-35388, STARLINK-33611, STARLINK-3277..."
5,sat_ranks,"[1, 2, 3, 4, 5]"
6,sampling_frequency_hz,1.0
7,num_time_steps,1
8,jam_precoder_mode,joint_waterfilling
9,jam_precoder_num_streams,None


,key,present,shape,ndim,dtype,size,nbytes_MB
0,a_tx,True,"(5, 36, 1, 36, 1, 1)",6.0,complex128,6480,0.103680
1,tau_tx,True,"(5, 1, 1)",3.0,float64,5,0.000040
2,a_jam,True,"(5, 36, 1, 36, 1, 1)",6.0,complex128,6480,0.103680
3,tau_jam,True,"(5, 1, 1)",3.0,float64,5,0.000040
4,a_jam_eff,True,"(5, 36, 1, 5, 1, 1)",6.0,complex128,900,0.014400
5,W_jam,True,"(36, 5)",2.0,complex128,180,0.002880
6,w_jam,True,"(36, 1)",2.0,complex128,36,0.000576
7,a_eff,False,None,NaN,NaN,0,NaN
8,W_t,False,None,NaN,NaN,0,NaN
9,w_t,False,None,NaN,NaN,0,NaN
